# DAPFAM Data Review

สมุดงานนี้อ่านเฉพาะ safe batch แบบ hash/count-only จาก Owner-local store และไม่เปิด raw membership, qrels payload หรือผลราย query

DAPFAM measures family-level retrieval relevance. It does not establish novelty, validity, infringement, freedom to operate, or legal truth.

In [ ]:
import hashlib, json, os
from pathlib import Path

def discover_safe_batch():
    configured = os.environ.get('MYIS_F1G1_SAFE_BATCH')
    if configured:
        configured_path = Path(configured)
        if configured_path.is_symlink():
            raise RuntimeError('MYIS_F1G1_SAFE_BATCH must identify a regular file')
        candidate = configured_path.resolve(strict=True)
        if not candidate.is_file():
            raise RuntimeError('MYIS_F1G1_SAFE_BATCH must identify a regular file')
        return candidate

    cwd = Path.cwd().resolve(strict=True)
    for ancestor in (cwd, *cwd.parents):
        projection_path = ancestor / '01_Stores' / '00_myIS' / 'owner-local' / 'f1-g1' / 'safe' / 'projections' / 'current.json'
        if not projection_path.is_file() or projection_path.is_symlink():
            continue
        projection = json.loads(projection_path.read_text(encoding='utf-8'))
        if projection.get('schema_version') != 'myis.f1-g1-safe-projection.v1':
            raise RuntimeError('Owner-local F1/G1 projection schema is invalid')
        batch_id = projection.get('safe_batch_id')
        if not isinstance(batch_id, str) or Path(batch_id).name != batch_id:
            raise RuntimeError('Owner-local F1/G1 projection batch ID is invalid')
        batch_root = projection_path.parents[1] / 'batches'
        unresolved = batch_root / batch_id
        if unresolved.is_symlink():
            raise RuntimeError('Owner-local F1/G1 safe batch is invalid')
        candidate = unresolved.resolve(strict=True)
        if not candidate.is_file() or candidate.parent != batch_root.resolve(strict=True):
            raise RuntimeError('Owner-local F1/G1 safe batch is invalid')
        expected_sha256 = projection.get('safe_batch_sha256')
        if not isinstance(expected_sha256, str) or len(expected_sha256) != 64:
            raise RuntimeError('Owner-local F1/G1 safe batch hash is invalid')
        actual_sha256 = hashlib.sha256(candidate.read_bytes()).hexdigest()
        if actual_sha256 != expected_sha256:
            raise RuntimeError('Owner-local F1/G1 safe batch hash mismatch')
        return candidate
    raise RuntimeError('Owner-local F1/G1 safe projection not found; run Start-F1G1Preparation.ps1 first')

safe_batch = discover_safe_batch()
batch = json.loads(safe_batch.read_text(encoding='utf-8'))
assert batch['schema_version'] == 'myis.g1-owner-value-batch.v1'
assert batch['gate_status'] == 'pending'
assert batch['authorization'] == 'NOT_AUTHORIZED'
assert batch['scientific_run'] is False
assert batch['scientific_metric_count'] == 0
'Safe preparation batch validated'

In [ ]:
counts = batch['inventory_counts']
split = batch['split']['counts']
print(f"Corpus: {counts['corpus']:,} | Queries: {counts['queries']:,} | Qrels: {counts['qrels']:,}")
print(f"Fresh split (seed 42): train {split['train']} | selection {split['selection']} | joint test {split['joint_test']}")
print('OUT-positive by split:', batch['split']['out_positive_counts'])

In [ ]:
domains = batch['qrels_domain_distribution']
peak = max(domains.values()) if domains else 1
for domain in ('IN', 'OUT', 'NC'):
    value = domains.get(domain, 0)
    bar = '|' * round(40 * value / peak)
    print(f"{domain:>3} {value:>6,} {bar}")

## B0 / B1 / B2 aggregate views

**ยังไม่รัน - รอ G1**

Future measured aggregate views may be added only after a valid G1 decision and frozen RunSpec. This preparation notebook contains no scientific metrics.